# Example showing how the caching works
- A change in estimator causes a fresh run
- An identical estimator (even if technically a different instance) will use cached data
- A fresh run always happens on the first run
- If the input data is different, the cached data is ignore.
- This works across runs i.e. if you run this notebook again, all fit and predict calls will be quick because the data will have been saved. Delete the data in `./cache_data` for it to run as new.
- We use a `RandomForestClassifier`, try different estimators or even a custom one with a slower fit/predict and you'll see even greater improvements.
- This is a simple example, anything you can do with estimators you should be able to do with the wrapped estimator (use in `Pipeline`s, crossval etc.)
- We've used numpy arrays, but it should also work nicely with pandas and polars data.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath('../'))

In [ ]:
import logging
import structlog

# Set the logging level for standard logging
logging.basicConfig(level=logging.WARNING)

# Configure structlog to use the standard logging
structlog.configure(
    processors=[
        structlog.processors.KeyValueRenderer(),
    ],
    context_class=dict,
    logger_factory=structlog.stdlib.LoggerFactory(),
    wrapper_class=structlog.stdlib.BoundLogger,
)

# Set the logging level for structlog as well
logging.getLogger().setLevel(logging.WARNING)

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

from sklearn_estimator_caching import EstimatorCachingWrapperGetter

In [ ]:
wrapper = EstimatorCachingWrapperGetter(
    # if we modify experiments or project (or any other kwarg of your choice) then the data will be saved/loaded from a new location.
    project="sklearn_wrapping_development",
    experiment="testing1",
    base_dir="./cache_data"
)

In [ ]:
# large dataset so that fit/predict take some time. Increase these numbers and you'll see an even greated speedup.
X, y = make_classification(n_samples=100_000, n_features=10,
                           n_informative=2, n_redundant=0,
                           random_state=0, shuffle=False)

In [ ]:
random_forest = wrapper(
    RandomForestClassifier()
)
random_forest # note how the output retains the detail of the original classifier.


### First fit call
Should take a while since we have a a lot of data

In [ ]:
%%time
random_forest.fit(X, y)

### Second fit call
Should be very quick, since the estimator and the data are the same, we load from the cache.

In [ ]:
%%time
random_forest.fit(X, y)

### Change the training data
With the same estimator but different training data, the fit should run again from scratch.

In [ ]:
X2 = X.copy()
np.random.shuffle(X2)

In [ ]:
%%time
random_forest.fit(X2, y)

### Modify the target data, the fit runs again from scratch

In [ ]:
y2 = y.copy()
np.random.shuffle(y2)

In [ ]:
%%time
random_forest.fit(X2, y2)

In [ ]:
%%time
random_forest.fit(X2, y2)

### A different but identical wrapped classifier, and the same input data, we use the cached estimator
i.e. a different instance with the exact same args/kwargs (in this case none) will behave as the original one, seen as the data processing should be identical.

In [ ]:
%%time
wrapper(RandomForestClassifier()).fit(X, y)

### The same input data, but a different estimator - with some different attribute
Runs the fit again as it is a different model

In [ ]:
%%time
wrapper(RandomForestClassifier(random_state = 7)).fit(X, y)

## Predict calls
FYI these examples would work the same with a transformer

### The first call runs the prediction

In [ ]:
%%time
predictions_first_call = random_forest.predict(X)

### The second call with the same data loads the cached data instead of running the prediction, and the outputs are identical

In [ ]:
%%time
predictions_second_call = random_forest.predict(X)

In [ ]:
np.testing.assert_array_equal(predictions_first_call, predictions_second_call)

### Using different data means that it re-runs the prediction

In [ ]:
%%time
predictions_third_call = random_forest.predict(X2)

In [ ]:
%%time
predictions_fourth_call = random_forest.predict(X2)

In [ ]:
assert not np.array_equal(predictions_second_call, predictions_third_call)
np.testing.assert_array_equal(predictions_third_call, predictions_fourth_call)

### All of the behaviour shown also works with fit_predict (and fit_transform in the case of transformers)
because both fit and predict have been run and cached, `fit_predict` will use that cached data.

In [ ]:
%%time
predictions_fifth_call = random_forest.fit_predict(X, y)

In [ ]:
assert not np.array_equal(predictions_fifth_call, predictions_third_call)
np.testing.assert_array_equal(predictions_second_call, predictions_fifth_call)